# NotebookLM Kit — Notebook Inventory

Read-only view of a notebook's contents.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Sources** — list all sources (indexed)
4. **Artifacts** — list all artifacts with their type, creation time, and the indices of the sources used to generate each one (referencing the table above)

In [ ]:
import importlib, config, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(config)
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from config import NOTEBOOK_ID
from pipeline import load_credentials, check_tsx, login

login()

creds = load_credentials(mode="auto")
check_tsx()


credentials: using patchright (credentials.json found)
Credentials ready — token: 42 chars, cookies: 2548 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
# NOTEBOOK_ID = "0a79eb08-bbdb-48a0-9b23-5a2ca44e368f"  # ← paste your notebook ID here

## Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources.  
The index column is what the **Artifacts** table below uses to reference each source.

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)


Notebook :  4 Napoleon
+---+----------------------+------------+--------------------------------------+
| # | Title                | Status     | Source ID                            |
+---+----------------------+------------+--------------------------------------+
| 0 | 00. Introduction.txt | READY      | 89fd686f-2cd5-4ef0-b427-50424c8d24b4 |
| 1 | 01. Corsica.txt      | READY      | ce3a771d-70b1-44ee-949c-8e83e4a21d52 |
| 2 | 02. Revolution.txt   | READY      | ebd71561-b799-4de4-bde9-1086cef8ae7e |
| 3 | 03. Desire.txt       | READY      | 5f2ccf0c-095a-499a-889f-9abb1bc92525 |
| 4 | 04. Italy.txt        | READY      | 56a7c070-e2d7-41f0-8467-5fe755e38f42 |
| 5 | 05. Victory.txt      | READY      | 36ac048c-22b5-403d-be4e-8d80f1cc68fc |
| 6 | 06. Peace.txt        | READY      | 3677333b-9223-4605-8920-f0b914ab52a5 |
| 7 | 07. Egypt.txt        | READY      | 8606ebfd-a9fb-443e-9779-cda83dac7c94 |
| 8 | 08. Acre.txt         | READY      | 5352bfee-94aa-4f22-bff3-7f9984c43aa6 |
| 9 

## Artifacts

`list_artifacts(notebook_id, sources, creds)` — every artifact in the notebook with:

- **Title** and **Type** (FLASHCARDS, QUIZ, VIDEO, …)
- **Created** — local time pulled from the artifact's own `createdAt`
- **Sources** — comma-separated **indices** into the sources table above (e.g. `0,2` means rows 0 and 2)

In [ ]:
from pipeline import list_artifacts

artifacts = list_artifacts(NOTEBOOK_ID, sources, creds)


Artifacts in notebook 0a79eb08-bbdb-48a0-9b23-5a2ca44e368f (35 total)
+----+----------------------------------+-------------+---------------------+---------+--------------------------------------+
| #  | Title                            | Type        | Created             | Sources | Artifact ID                          |
+----+----------------------------------+-------------+---------------------+---------+--------------------------------------+
|  0 | Consolidation to Conquest        | VIDEO       | 2026-05-31 16:33:25 | 17      | f56f6788-7e5f-42d1-ad0f-3b657b9f9adc |
|  1 | Napoleon's Empire: 1809-1810     | VIDEO       | 2026-05-31 16:33:36 | 22      | c792a615-ecf1-4709-9c41-7eb35bbe578b |
|  2 | The Retreat from Moscow          | VIDEO       | 2026-05-31 16:33:43 | 25      | 9fcd3739-9034-4634-bd41-716c0e294a2e |
|  3 | Napoleon: Empire & Iberia        | VIDEO       | 2026-05-31 16:33:32 | 20      | 991b0c2a-c319-45ff-9ec6-ff9c006515c4 |
|  4 | Napoleon's 1813 Crisis           

## Rename Single-Source Artifacts

`rename_single_source_artifacts(artifacts, sources, creds, *, indices=None, dry_run=False)`

For every artifact whose `sourceIds` contains exactly **one** source, rename it to:

```
<source title> YYMMDD HHMM
```

where the timestamp comes from the artifact's own `createdAt` (local time).

- `indices` — optional list of row numbers from the Artifacts table above to limit the operation; `None` means all single-source artifacts.
- `dry_run=True` — preview the planned renames without calling the API.

Multi-source artifacts (e.g. audio overviews built from all sources) are left untouched.

In [ ]:
from pipeline import rename_single_source_artifacts

# Preview only — no changes made
rename_single_source_artifacts(artifacts, sources, creds, dry_run=False)

# Limit to specific rows from the Artifacts table:
# rename_single_source_artifacts(artifacts, sources, creds, indices=[0, 1, 3], dry_run=True)


Renaming 19 single-source artifact(s)
+----+-------------------------------+--------------------------------+----------+
| #  | Old title                     | New title                      | Status   |
+----+-------------------------------+--------------------------------+----------+
|  0 | Consolidation to Conquest     | 17. Jena [260531 163325]       | ok       |
|  1 | Napoleon's Empire: 1809-1810  | 22. Zenith [260531 163336]     | ok       |
|  2 | The Retreat from Moscow       | 25. Retreat [260531 163343]    | ok       |
|  3 | Napoleon: Empire & Iberia     | 20. Iberia [260531 163332]     | ok       |
|  4 | Napoleon's 1813 Crisis        | 26. Resilience [260531 163345] | ok       |
|  5 | Fall of Napoleon's Empire     | 27. Leipzig [260531 163347]    | ok       |
|  6 | Napoleon's Defeat at Waterloo | 30. Waterloo [260531 163353]   | ok       |
|  7 | The 1809 Danube Campaign      | 21. Wagram [260531 163334]     | ok       |
|  8 | The 1814 Campaign             | 28. Defia

[{'index': 0,
  'artifactId': 'f56f6788-7e5f-42d1-ad0f-3b657b9f9adc',
  'oldTitle': 'Consolidation to Conquest',
  'newTitle': '17. Jena [260531 163325]',
  'status': 'ok',
  'error': None},
 {'index': 1,
  'artifactId': 'c792a615-ecf1-4709-9c41-7eb35bbe578b',
  'oldTitle': "Napoleon's Empire: 1809-1810",
  'newTitle': '22. Zenith [260531 163336]',
  'status': 'ok',
  'error': None},
 {'index': 2,
  'artifactId': '9fcd3739-9034-4634-bd41-716c0e294a2e',
  'oldTitle': 'The Retreat from Moscow',
  'newTitle': '25. Retreat [260531 163343]',
  'status': 'ok',
  'error': None},
 {'index': 3,
  'artifactId': '991b0c2a-c319-45ff-9ec6-ff9c006515c4',
  'oldTitle': 'Napoleon: Empire & Iberia',
  'newTitle': '20. Iberia [260531 163332]',
  'status': 'ok',
  'error': None},
 {'index': 4,
  'artifactId': 'b99034f0-66e7-4986-a6d9-a0fdd8e05ccd',
  'oldTitle': "Napoleon's 1813 Crisis",
  'newTitle': '26. Resilience [260531 163345]',
  'status': 'ok',
  'error': None},
 {'index': 5,
  'artifactId': '593

## Download Artifacts by Type

download_artifacts_by_type(artifacts, artifact_type, notebook_id, creds, *, indices=None, output_dir=None)

Downloads every artifact of the given type from the rtifacts list (or a subset via indices).
Files land in outputs/<artifact_type_lower>/<Notebook Name>/<yyyymmdd_hhmmss>_<title><ext>.
Already-downloaded files are skipped automatically.

Valid types: FLASHCARDS, QUIZ, AUDIO, INFOGRAPHIC, VIDEO, SLIDE_DECK.

In [ ]:
from pipeline import download_artifacts_by_type

download_artifacts_by_type(artifacts, "Infographic", NOTEBOOK_ID, creds)

# Limit to specific rows from the Artifacts table:
# download_artifacts_by_type(artifacts, "VIDEO", NOTEBOOK_ID, creds, indices=[0, 2, 5])
# download_artifacts_by_type(artifacts[2:6], "AUDIO", NOTEBOOK_ID, creds)


Downloaded 3 / 3 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\infographic\4 Napoleon
+---+----------------------------------+----------------------------------------------+------------+
| # | Source                           | File                                         | Status     |
+---+----------------------------------+----------------------------------------------+------------+
| 0 | 00. Introduction [260525 103420] | 00. Introduction 260525 103420.json          | downloaded |
| 1 | 01. Corsica [260525 103444]      | 01. Corsica 260525 103444.json               | downloaded |
| 2 | 02. Revolution [260525 103501]   | 02. Revolution 260525 103501.json            | downloaded |
+---+----------------------------------+----------------------------------------------+------------+


[{'sourceTitle': '00. Introduction [260525 103420]',
  'notebookTitle': ' 4 Napoleon',
  'createdAt': '2026-05-25T05:04:20.919Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\4 Napoleon\\00. Introduction 260525 103420.json',
  'status': 'downloaded'},
 {'sourceTitle': '01. Corsica [260525 103444]',
  'notebookTitle': ' 4 Napoleon',
  'createdAt': '2026-05-25T05:04:44.286Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\4 Napoleon\\01. Corsica 260525 103444.json',
  'status': 'downloaded'},
 {'sourceTitle': '02. Revolution [260525 103501]',
  'notebookTitle': ' 4 Napoleon',
  'createdAt': '2026-05-25T05:05:01.818Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\4 Napoleon\\02. Revolution 260525 103501.json',
  'status': 'downloaded'}]